# 01 — NLP / Local Evaluators

**What you'll learn:**
- The deterministic, no-LLM evaluators and what each measures
- When token-overlap metrics are (and aren't) the right tool

---

These evaluators compare `response` against `ground_truth` using classic NLP
overlap math. They are **fast, free, and reproducible** — no judge model, no API
cost — which makes them great for CI smoke tests. Their weakness: they reward
surface wording, not meaning, so a correct paraphrase can score low.

In [1]:
import os
import site
import shutil
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# --------------------------------------------------------------------------
# Reasoning-model compatibility (gpt-5.x / o-series judge models)
# --------------------------------------------------------------------------
# Reasoning models reject `max_tokens` (they require `max_completion_tokens`) and
# only allow the default temperature. azure-ai-evaluation runs evaluators in
# spawned worker processes, so the fix must load in every worker: we copy the
# workshop's `sitecustomize.py` shim into site-packages (auto-imported by every
# Python process) and also apply it here in the kernel. It is a no-op for classic
# models (gpt-4o etc.), so it is always safe to run.
_shim_src = Path("../sitecustomize.py")
if _shim_src.exists():
    _shim_dst = Path(site.getsitepackages()[0]) / "sitecustomize.py"
    if not _shim_dst.exists() or _shim_dst.read_text() != _shim_src.read_text():
        shutil.copyfile(_shim_src, _shim_dst)
    exec(_shim_src.read_text())

# Judge model config for AI-assisted evaluators (notebooks 02, 04, 05).
# Uses az login / DefaultAzureCredential when AZURE_OPENAI_API_KEY is blank.
model_config = {
    "azure_endpoint": os.environ["AZURE_OPENAI_ENDPOINT"],
    "azure_deployment": os.environ["AZURE_OPENAI_DEPLOYMENT"],
    "api_version": os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
}
if os.environ.get("AZURE_OPENAI_API_KEY"):
    model_config["api_key"] = os.environ["AZURE_OPENAI_API_KEY"]

# Foundry project (notebooks 03, 06) — enables cloud logging + safety evaluators.
azure_ai_project = os.environ.get("AZURE_AI_PROJECT_ENDPOINT")

DATA_DIR = "../data"
print("Judge deployment:", model_config["azure_deployment"])

Judge deployment: gpt-5.4-mini


## The metrics

| Evaluator | Measures | Typical use |
|-----------|----------|-------------|
| `F1ScoreEvaluator` | token precision/recall overlap | short factual answers |
| `BleuScoreEvaluator` | n-gram precision (translation heritage) | translation, tight paraphrase |
| `RougeScoreEvaluator` | n-gram recall | summarization |
| `MeteorScoreEvaluator` | overlap with stemming + synonyms | more lenient than BLEU |
| `GleuScoreEvaluator` | sentence-level BLEU variant | shorter texts |

In [2]:
from azure.ai.evaluation import (
    evaluate,
    F1ScoreEvaluator,
    BleuScoreEvaluator,
    RougeScoreEvaluator,
    MeteorScoreEvaluator,
    GleuScoreEvaluator,
)
from azure.ai.evaluation import RougeType

evaluators = {
    "f1": F1ScoreEvaluator(),
    "bleu": BleuScoreEvaluator(),
    "rouge": RougeScoreEvaluator(rouge_type=RougeType.ROUGE_L),
    "meteor": MeteorScoreEvaluator(),
    "gleu": GleuScoreEvaluator(),
}

In [3]:
result = evaluate(
    data=f"{DATA_DIR}/insurance-qa.jsonl",
    evaluators=evaluators,
)
result["metrics"]

2026-07-21 10:20:18 -0400 140106062771904 execution.bulk     INFO     Finished 8 / 8 lines.
2026-07-21 10:20:18 -0400 140106045986496 execution.bulk     INFO     Finished 8 / 8 lines.
2026-07-21 10:20:18 -0400 140106062771904 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.
2026-07-21 10:20:18 -0400 140106045986496 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.
2026-07-21 10:20:18 -0400 140106054379200 execution.bulk     INFO     Finished 8 / 8 lines.
2026-07-21 10:20:18 -0400 140106054379200 execution.bulk     INFO     Average execution time for completed lines: 0.02 seconds. Estimated time for incomplete lines: 0.0 seconds.
2026-07-21 10:20:18 -0400 140106029201088 execution.bulk     INFO     Finished 8 / 8 lines.
2026-07-21 10:20:18 -0400 140106029201088 execution.bulk     INFO     Average execution time for com

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "f1_20260721_142018_784808"
Run status: "Completed"
Start time: "2026-07-21 14:20:18.784808+00:00"
Duration: "0:00:01.006049"

======= Run Summary =======

Run name: "rouge_20260721_142018_791492"
Run status: "Completed"
Start time: "2026-07-21 14:20:18.791492+00:00"
Duration: "0:00:01.005225"

======= Run Summary =======

Run name: "bleu_20260721_142018_789713"
Run status: "Completed"
Start time: "2026-07-21 14:20:18.789713+00:00"
Duration: "0:00:01.007103"

======= Run Summary =======

Run name: "gleu_20260721_142018_796722"
Run status: "Completed"
Start time: "2026-07-21 14:20:18.796722+00:00"
Duration: "0:00:01.006197"

2026-07-21 10:20:22 -0400 140106037593792 execution.bulk     INFO     Finished 8 / 8 lines.
2026-07-21 10:20:22 -0400 140106037593792 execution.bulk     INFO     Average execution time for completed lines: 0.41 seconds. Estimated time for incomplete lines: 0.0 seconds.


Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "meteor_20260721_142018_794845"
Run status: "Completed"
Start time: "2026-07-21 14:20:18.794845+00:00"
Duration: "0:00:03.320934"

======= Combined Run Summary (Per Evaluator) =======

{
    "f1": {
        "status": "Completed",
        "duration": "0:00:01.006049",
        "completed_lines": 8,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "bleu": {
        "status": "Completed",
        "duration": "0:00:01.007103",
        "completed_lines": 8,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "rouge": {
        "status": "Completed",
        "duration": "0:00:01.005225",
        "completed_lines": 8,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": n

{'f1.f1_score': 0.49649287190471236,
 'f1.f1_score_score': 0.49649287190471236,
 'f1.f1_score_passed': 0.5,
 'bleu.bleu': 0.18482164649136723,
 'bleu.bleu_score': 0.18482164649136723,
 'bleu.bleu_passed': 0.0,
 'rouge.rouge': 0.49166313775117454,
 'rouge.rouge_score': 0.49166313775117454,
 'rouge.rouge_passed': 0.5,
 'meteor.meteor': 0.6396882374173951,
 'meteor.meteor_score': 0.6396882374173951,
 'meteor.meteor_passed': 0.75,
 'gleu.gleu': 0.2349814027430376,
 'gleu.gleu_score': 0.2349814027430376,
 'gleu.gleu_passed': 0.0,
 'f1.binary_aggregate': 0.5,
 'bleu.binary_aggregate': 0.0,
 'rouge.binary_aggregate': 0.5,
 'meteor.binary_aggregate': 0.75,
 'gleu.binary_aggregate': 0.0}

Version mismatch between change and config responses.No configurations updated.
Version mismatch between change and config responses.No configurations updated.


## Reading the spread

Because these metrics score wording overlap, a factually-correct-but-reworded answer
can score low. Inspect per-row values to see where overlap is weak — a low score is a
*prompt to look*, not proof of a wrong answer.

In [4]:
import pandas as pd

rows = pd.DataFrame(result["rows"])
score_cols = [c for c in rows.columns if c.endswith("_score") or c.endswith("score")]
rows[["inputs.query"] + [c for c in rows.columns if c.startswith("outputs.")]].head()

,inputs.query,outputs.f1.f1_score,outputs.f1.f1_score_score,outputs.f1.f1_score_passed,outputs.f1.f1_score_result,outputs.f1.f1_score_reason,outputs.f1.f1_score_status,outputs.f1.f1_score_threshold,outputs.f1.f1_score_properties,outputs.bleu.bleu,...,outputs.meteor.meteor_threshold,outputs.meteor.meteor_properties,outputs.gleu.gleu,outputs.gleu.gleu_score,outputs.gleu.gleu_passed,outputs.gleu.gleu_result,outputs.gleu.gleu_reason,outputs.gleu.gleu_status,outputs.gleu.gleu_threshold,outputs.gleu.gleu_properties
0,How long do I have to file an auto claim after...,0.375000,0.375000,False,fail,None,completed,0.5,None,0.147101,...,0.5,None,0.163265,0.163265,False,fail,None,completed,0.5,None
1,Does my homeowners policy cover water damage f...,0.606061,0.606061,True,pass,None,completed,0.5,None,0.045686,...,0.5,None,0.144444,0.144444,False,fail,None,completed,0.5,None
2,What is a deductible?,0.413793,0.413793,False,fail,None,completed,0.5,None,0.187737,...,0.5,None,0.232558,0.232558,False,fail,None,completed,0.5,None
3,Can I add my teenage child to my auto policy?,0.363636,0.363636,False,fail,None,completed,0.5,None,0.094977,...,0.5,None,0.179245,0.179245,False,fail,None,completed,0.5,None
4,Is flood damage covered under a standard homeo...,0.608696,0.608696,True,pass,None,completed,0.5,None,0.284333,...,0.5,None,0.303030,0.303030,False,fail,None,completed,0.5,None


## Recap

- NLP evaluators are deterministic and free — ideal for fast CI gates.
- They measure *wording overlap*, not *meaning* — pair them with AI-assisted metrics.

**Next:** [02 — AI-assisted quality](./02-ai-assisted-quality.ipynb) — LLM-judged metrics
that understand meaning.